# Dataset Characterization

> *Statistics on the dataset used in this study: projects, PRs, merges,
> conflicting merges, and conflicting chunks. Results feed into the
> paper's §4.1 "Dataset Overview".*

We report two layers:

* **The AIDev universe** — the starting point, reproduced from the
  AIDev parquets (`all_pull_request.parquet`, `all_repository.parquet`,
  `pr_task_type.parquet`).
* **Our analysis subset** — what survives Pass A (chronology) and
  Pass B (merge extraction, conflict detection, resolution
  localization), deduplicated to the physical merge level via
  `(repo_full_name, merge_sha)`.

All chunk-level counts use the natural key
`(repo_full_name, merge_sha, file_path, chunk_index)` to avoid the
inflation introduced by a single merge commit being referenced from
several PRs (force-push / re-open / cherry-pick).

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
for candidate in [_here, *_here.parents]:
    if (candidate / "analysis" / "common.py").exists():
        PROJECT_ROOT = candidate
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from analysis.common import (
    load_tables, pr_fanout, setup_style, save_fig,
    DATA_DIR,
)

setup_style()
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

AIDEV_DIR = PROJECT_ROOT / "AIDev"

## 1. The AIDev universe (before any of our processing)

Baseline counts from the AIDev distribution itself, so the reader can
see how much of AIDev we cover.

In [ ]:
pr_aidev = pd.read_parquet(AIDEV_DIR / "all_pull_request.parquet")
repo_aidev = pd.read_parquet(AIDEV_DIR / "all_repository.parquet")

print(f"AIDev PRs:           {len(pr_aidev):,}")
print(f"AIDev repositories:  {len(repo_aidev):,}")
print(f"AIDev users (unique repo authors): {pr_aidev['user'].nunique():,}")

print("\nAIDev PRs by agent:")
print(pr_aidev['agent'].value_counts().to_string())

print("\nAIDev PRs by state:")
print(pr_aidev['state'].value_counts().to_string())

In [ ]:
# Language mix at the repository level (AIDev)
lang_counts = repo_aidev['language'].fillna('Unknown').replace('', 'Unknown').value_counts()
print("AIDev repositories by language (top 15):")
print(lang_counts.head(15).to_string())
print(f"\nTotal distinct languages: {lang_counts.size}")

## 2. Our analysis subset

What survives Pass A (chronology) plus Pass B (merge extraction,
conflict detection, resolution localization), **deduplicated** to the
physical merge and physical chunk.

`load_tables` already collapses the pipeline outputs on the natural
keys, so the counts below are on physical artifacts, not on
`(pr_id, merge_sha)` pairs.

In [ ]:
tables = load_tables(deduplicate=True)
tables_raw = load_tables(deduplicate=False)

universe = tables.universe
im       = tables.internal_merges          # deduplicated (repo, merge_sha)
im_raw   = tables_raw.internal_merges      # one row per (pr_id, merge_sha)
cc       = tables.classified_chunks        # deduplicated natural key
cc_raw   = tables_raw.classified_chunks
err      = tables.extraction_errors

print("=== Raw vs. deduplicated table sizes ===")
print(f"internal_merges   raw: {len(im_raw):>10,}   dedup: {len(im):>10,}")
print(f"classified_chunks raw: {len(cc_raw):>10,}   dedup: {len(cc):>10,}")

In [ ]:
# Headline numbers for the "Dataset Overview" section of the paper.
projects_in_scope = universe['full_name'].nunique() if not universe.empty else 0
prs_in_scope      = universe['pr_id'].nunique()     if not universe.empty else 0

internal_merges_total = len(im)
projects_with_merges  = im['repo_full_name'].nunique() if not im.empty else 0

# A merge "has conflict" iff it appears in classified_chunks.
conflict_merge_keys = (
    cc[['repo_full_name', 'merge_sha']].drop_duplicates() if not cc.empty else pd.DataFrame()
)
conflicting_merges = len(conflict_merge_keys)
total_chunks = len(cc)

# PRs that contain at least one internal merge
prs_with_internal_merge = im_raw['pr_id'].nunique() if not im_raw.empty else 0
# PRs that contain at least one conflicting merge (via the raw tables)
prs_with_conflict = cc_raw['pr_id'].nunique() if not cc_raw.empty else 0

def pct(a, b):
    return f"{(a / b * 100):.2f}%" if b else "n/a"

summary = pd.DataFrame([
    ("Projects in scope",            projects_in_scope, len(repo_aidev), pct(projects_in_scope, len(repo_aidev))),
    ("PRs in scope",                 prs_in_scope,      len(pr_aidev),   pct(prs_in_scope, len(pr_aidev))),
    ("PRs with >=1 internal merge",  prs_with_internal_merge, prs_in_scope, pct(prs_with_internal_merge, prs_in_scope)),
    ("PRs with >=1 conflict",        prs_with_conflict,       prs_in_scope, pct(prs_with_conflict, prs_in_scope)),
    ("Projects with internal merges", projects_with_merges,    projects_in_scope, pct(projects_with_merges, projects_in_scope)),
    ("Internal merges (dedup)",      internal_merges_total,   None, ""),
    ("Conflicting merges",           conflicting_merges,      internal_merges_total, pct(conflicting_merges, internal_merges_total)),
    ("Conflicting chunks (dedup)",   total_chunks,            None, ""),
], columns=["Metric", "Count", "Denominator", "Share"])

print(summary.to_string(index=False))

## 3. Per-agent breakdown

One row per agent with: PRs in scope, PRs with merges, PRs with
conflicts, internal merges, conflicting merges, and chunks.

In [ ]:
pr_ctx_cols = [c for c in ('pr_id', 'full_name', 'agent', 'language', 'pr_task_type', 'state', 'merged_at') if c in universe.columns]
pr_ctx = universe[pr_ctx_cols].drop_duplicates(subset=['pr_id'])

# PRs per agent (scope)
prs_per_agent = pr_ctx.groupby('agent')['pr_id'].nunique().rename('prs')
projects_per_agent = pr_ctx.groupby('agent')['full_name'].nunique().rename('projects')

# PRs with internal merges per agent
im_with_agent = im_raw.merge(pr_ctx[['pr_id', 'agent']], on='pr_id', how='left')
prs_with_merge_per_agent = im_with_agent.groupby('agent')['pr_id'].nunique().rename('prs_with_merge')

# PRs with conflicts per agent
cc_with_agent = cc_raw.merge(pr_ctx[['pr_id', 'agent']], on='pr_id', how='left') if not cc_raw.empty else pd.DataFrame()
prs_with_conflict_per_agent = (
    cc_with_agent.groupby('agent')['pr_id'].nunique().rename('prs_with_conflict')
    if not cc_with_agent.empty else pd.Series(dtype=int, name='prs_with_conflict')
)

# Internal merges per agent (dedup at merge level, attributed via representative pr_id)
im_dedup_agent = im.merge(pr_ctx[['pr_id', 'agent']], on='pr_id', how='left')
merges_per_agent = im_dedup_agent.groupby('agent').size().rename('internal_merges')

# Conflicting merges per agent
cc_dedup_merge = cc[['repo_full_name', 'merge_sha']].drop_duplicates()
cc_dedup_merge = cc_dedup_merge.merge(im[['repo_full_name', 'merge_sha', 'pr_id']],
                                      on=['repo_full_name', 'merge_sha'], how='left')
cc_dedup_merge = cc_dedup_merge.merge(pr_ctx[['pr_id', 'agent']], on='pr_id', how='left')
conf_merges_per_agent = cc_dedup_merge.groupby('agent').size().rename('conflicting_merges')

# Chunks per agent
cc_agent = cc.merge(pr_ctx[['pr_id', 'agent']], on='pr_id', how='left')
chunks_per_agent = cc_agent.groupby('agent').size().rename('conflict_chunks')

by_agent = pd.concat([
    projects_per_agent, prs_per_agent, prs_with_merge_per_agent,
    prs_with_conflict_per_agent, merges_per_agent, conf_merges_per_agent,
    chunks_per_agent,
], axis=1).fillna(0).astype(int).sort_values('prs', ascending=False)

print(by_agent.to_string())

## 4. Per-language breakdown (top N)

Same logic as the per-agent table, with languages aggregated at the
repository level. Languages outside the top 10 are folded into
`Other` for readability.

In [ ]:
pr_ctx2 = pr_ctx.copy()
pr_ctx2['language'] = pr_ctx2['language'].fillna('Unknown').replace('', 'Unknown')
top_langs = pr_ctx2['language'].value_counts().head(10).index
pr_ctx2['language_top'] = pr_ctx2['language'].where(pr_ctx2['language'].isin(top_langs), other='Other')

projects_per_lang = pr_ctx2.groupby('language_top')['full_name'].nunique().rename('projects')
prs_per_lang = pr_ctx2.groupby('language_top')['pr_id'].nunique().rename('prs')

im_with_lang = im_raw.merge(pr_ctx2[['pr_id', 'language_top']], on='pr_id', how='left')
prs_with_merge_per_lang = im_with_lang.groupby('language_top')['pr_id'].nunique().rename('prs_with_merge')

cc_with_lang = cc_raw.merge(pr_ctx2[['pr_id', 'language_top']], on='pr_id', how='left') if not cc_raw.empty else pd.DataFrame()
prs_with_conflict_per_lang = (
    cc_with_lang.groupby('language_top')['pr_id'].nunique().rename('prs_with_conflict')
    if not cc_with_lang.empty else pd.Series(dtype=int, name='prs_with_conflict')
)

im_dedup_lang = im.merge(pr_ctx2[['pr_id', 'language_top']], on='pr_id', how='left')
merges_per_lang = im_dedup_lang.groupby('language_top').size().rename('internal_merges')

cc_dedup_lang = cc_dedup_merge.merge(pr_ctx2[['pr_id', 'language_top']], on='pr_id', how='left')
conf_merges_per_lang = cc_dedup_lang.groupby('language_top').size().rename('conflicting_merges')

cc_lang = cc.merge(pr_ctx2[['pr_id', 'language_top']], on='pr_id', how='left')
chunks_per_lang = cc_lang.groupby('language_top').size().rename('conflict_chunks')

by_lang = pd.concat([
    projects_per_lang, prs_per_lang, prs_with_merge_per_lang,
    prs_with_conflict_per_lang, merges_per_lang, conf_merges_per_lang,
    chunks_per_lang,
], axis=1).fillna(0).astype(int).sort_values('prs', ascending=False)

print(by_lang.to_string())

## 5. Per-PR-task-type breakdown

`pr_task_type` is only populated under `--pop-only` (AIDev-pop ships
this enriched table). Under full AIDev the column is mostly null; the
section below degrades gracefully.

In [ ]:
tt_available = 'pr_task_type' in pr_ctx.columns and pr_ctx['pr_task_type'].notna().any()
if not tt_available:
    print("pr_task_type is not populated in this run (full AIDev scope).")
else:
    prs_per_tt = pr_ctx.groupby('pr_task_type')['pr_id'].nunique().rename('prs')
    im_with_tt = im_raw.merge(pr_ctx[['pr_id', 'pr_task_type']], on='pr_id', how='left')
    prs_with_merge_per_tt = im_with_tt.groupby('pr_task_type')['pr_id'].nunique().rename('prs_with_merge')
    cc_with_tt = cc_raw.merge(pr_ctx[['pr_id', 'pr_task_type']], on='pr_id', how='left') if not cc_raw.empty else pd.DataFrame()
    prs_with_conflict_per_tt = (
        cc_with_tt.groupby('pr_task_type')['pr_id'].nunique().rename('prs_with_conflict')
        if not cc_with_tt.empty else pd.Series(dtype=int, name='prs_with_conflict')
    )
    cc_tt = cc.merge(pr_ctx[['pr_id', 'pr_task_type']], on='pr_id', how='left')
    chunks_per_tt = cc_tt.groupby('pr_task_type').size().rename('conflict_chunks')

    by_tt = pd.concat([
        prs_per_tt, prs_with_merge_per_tt, prs_with_conflict_per_tt, chunks_per_tt,
    ], axis=1).fillna(0).astype(int).sort_values('prs', ascending=False)
    print(by_tt.to_string())

## 6. Chunks-per-merge distribution (on deduplicated merges)

Quick sanity view; the formal RQ1 distribution lives in
`rq1_structural_nature.ipynb`.

In [ ]:
if not cc.empty:
    n_chunks_per_merge = (
        cc.groupby(['repo_full_name', 'merge_sha']).size().rename('n_chunks').reset_index()
    )
    desc = n_chunks_per_merge['n_chunks'].describe(percentiles=[.5, .75, .9, .95, .99]).round(2)
    print("Chunks per conflicting merge:")
    print(desc.to_string())
    print(f"\nTotal conflicting merges: {len(n_chunks_per_merge):,}")
    print(f"Total chunks:             {n_chunks_per_merge['n_chunks'].sum():,}")

## 7. Duplication audit

Quantifies how severe the "same merge commit appearing in multiple
PRs" pattern is. Reported for the paper's methodology footnote that
motivates the deduplication step.

In [ ]:
fanout = pr_fanout(im_raw)
print(f"Physical merge commits:      {len(fanout):,}")
print(f"Raw (pr_id, merge_sha) rows: {len(im_raw):,}")
print(f"Inflation factor:            {len(im_raw) / max(len(fanout), 1):.2f}x")
print("\nDistribution of #PRs per physical merge:")
print(fanout['n_prs'].describe(percentiles=[.5, .75, .9, .95, .99]).round(2).to_string())
print("\nTop 10 merges by fanout:")
print(fanout.sort_values('n_prs', ascending=False).head(10)[['repo_full_name', 'merge_sha', 'n_prs']].to_string(index=False))

## 8. Extraction-error audit

How many PRs / SHAs were dropped at each stage, by `error_type`. This
feeds the "threats to validity" discussion on coverage.

In [ ]:
if err.empty:
    print("No extraction errors recorded.")
else:
    print(err['error_type'].value_counts().to_string())

---
### Outputs for the paper

The numbers printed above feed directly into the paper's §4.1 "Dataset
Overview". The headline table comes from cell 6 (global summary); the
per-agent and per-language tables from cells 8 and 10; the
duplication audit from cell 14 supports the methodology footnote on
deduplication.